# ZAIC 2026 LifeMate — baseline bước 1 (transcript gold, chưa ASR)

**Cách dùng**
1. Runtime → Change runtime type → **GPU: A100 hoặc L4** (T4 cũng chạy được nhưng chậm, notebook tự chuyển sang float16).
2. Upload `lifemate_baseline.zip` vào Google Drive, thư mục `MyDrive/ZAIC/`. Dữ liệu: notebook tự tải từ link của BTC; nếu tải lỗi thì upload `train+public_test.zip` vào cùng thư mục.
3. Runtime → **Run all**. Mất khoảng 30–60 phút trên A100/L4.
4. Gửi lại file **`results.zip`** (tự tải về máy và lưu vào `MyDrive/ZAIC/`).

In [ ]:
# ==== CẤU HÌNH ====
MODEL     = "Qwen/Qwen3-4B-Instruct-2507"   # LLM < 5B
BACKEND   = "vllm"                          # đổi sang "hf" nếu cài vLLM lỗi (chậm hơn nhiều)
BATCH     = 32                              # batch cho dev; đo latency dùng batch 1 ở cell riêng
DATA_URL  = "https://dl-challenge.zalo.ai/2026/lifemate_argfzdgert/train+public_test.zip"
DRIVE_DIR = "/content/drive/MyDrive/ZAIC"
WORK      = "/content/work"                 # code
DATA_RAW  = "/content/data_raw"             # dữ liệu giải nén
RES       = "/content/results"              # mọi thứ cần gửi lại

In [ ]:
import subprocess, torch, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
cap = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if cap >= 8 else "float16"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("GPU:", GPU_NAME, "| dtype:", DTYPE)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type"
os.makedirs(RES, exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile, shutil, glob
os.makedirs(DRIVE_DIR, exist_ok=True)
code_zip = os.path.join(DRIVE_DIR, "lifemate_baseline.zip")
if not os.path.exists(code_zip):
    from google.colab import files
    print("Không thấy lifemate_baseline.zip trong Drive -> chọn file để upload")
    up = files.upload(); code_zip = "/content/" + next(iter(up))
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(code_zip).extractall(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# ==== DỮ LIỆU ====
import os, subprocess, zipfile, glob
data_zip = os.path.join(DRIVE_DIR, "train+public_test.zip")
if not os.path.exists(data_zip):
    data_zip = "/content/train+public_test.zip"
    if not os.path.exists(data_zip):
        r = subprocess.run(["wget", "-q", "-O", data_zip, DATA_URL])
        assert r.returncode == 0, "Tải dữ liệu lỗi -> upload train+public_test.zip vào " + DRIVE_DIR + " rồi chạy lại cell này"
if not glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True):
    zipfile.ZipFile(data_zip).extractall(DATA_RAW)
DATA = os.path.dirname(os.path.dirname(glob.glob(DATA_RAW + "/**/train/train.jsonl", recursive=True)[0]))
print("DATA root:", DATA, sorted(os.listdir(DATA)))
print(open(os.path.join(DATA, "VERSION")).read() if os.path.exists(os.path.join(DATA, "VERSION")) else "")

In [ ]:
%%capture pipout
!pip install -q vllm

In [ ]:
print(pipout.stdout[-2000:]); print(pipout.stderr[-2000:])
!pip list 2>/dev/null | grep -iE "^(vllm|torch|transformers) " | tee {RES}/versions.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv >> {RES}/versions.txt

## 1. Kiểm tra không cần model (phải ra 100.00 và ALL PASS)

In [ ]:
%cd {WORK}
!python -m lifemate.run --root {DATA} --split train --backend oracle --oracle-mistakes --out runs/oracle --batch 16 2>/dev/null
!python scripts/score.py --root {DATA} --split train --pred runs/oracle/submission.jsonl --by-tag | tee {RES}/sanity.txt
!python tests/test_guards.py --root {DATA} | tee -a {RES}/sanity.txt

## 2. Số token thật của prompt (giới hạn 16K)

In [ ]:
%cd {WORK}
!python -m lifemate.tokstats --root {DATA} --model {MODEL} 2>&1 | grep -v Warning | tee {RES}/tokstats.txt

## 3. Smoke test 5 mẫu: kiểm tra model có sinh `<tool_call>` đúng định dạng

In [ ]:
%cd {WORK}
!python -m lifemate.run --root {DATA} --split train --limit 5 --backend {BACKEND} --model {MODEL} --dtype {DTYPE} --out runs/smoke 2>&1 | tail -3
import json
with open(f"{RES}/smoke.txt", "w") as f:
    for l in open("runs/smoke/traces.jsonl"):
        t = json.loads(l); s = f"{t['id']}: " + json.dumps(t["events"][:6], ensure_ascii=False)[:1500]
        print(s); f.write(s + "\n")

## 4. Baseline trên val_unseen (app lạ) và val_seen

In [ ]:
%cd {WORK}
import time
for S in ["val_unseen", "val_seen"]:
    t = time.time()
    !python -m lifemate.run --root {DATA} --split train --ids splits/{S}.txt --backend {BACKEND} --model {MODEL} --dtype {DTYPE} --batch {BATCH} --out runs/base_{S} 2>&1 | tail -2
    print(S, "took", round(time.time() - t), "s")
    !python scripts/score.py --root {DATA} --split train --pred runs/base_{S}/submission.jsonl --by-tag | tee {RES}/score_{S}.txt
    !python -m lifemate.report --root {DATA} --split train --pred runs/base_{S}/submission.jsonl --show 6 > {RES}/report_{S}.txt
    !head -14 {RES}/report_{S}.txt

## 5. Latency từng mẫu với batch 1 (luật: ≤ 30 s/mẫu trên A30, gồm cả ASR)

In [ ]:
%cd {WORK}
!head -30 splits/val_unseen.txt > splits/lat30.txt
!python -m lifemate.run --root {DATA} --split train --ids splits/lat30.txt --backend {BACKEND} --model {MODEL} --dtype {DTYPE} --batch 1 --out runs/lat 2>&1 | tail -1
import json
secs = sorted(json.loads(l)["seconds"] for l in open("runs/lat/traces.jsonl"))
gens = [json.loads(l)["n_gen"] for l in open("runs/lat/traces.jsonl")]
s = f"GPU={GPU_NAME} batch=1 n={len(secs)} mean={sum(secs)/len(secs):.2f}s p50={secs[len(secs)//2]:.2f}s max={secs[-1]:.2f}s mean_gen_steps={sum(gens)/len(gens):.2f}"
print(s); open(f"{RES}/latency.txt", "w").write(s + "\n")

## 6. Đóng gói kết quả → gửi lại `results.zip`

In [ ]:
%cd {WORK}
import shutil, os
for S in ["val_unseen", "val_seen", "smoke"]:
    d = f"runs/base_{S}" if S != "smoke" else "runs/smoke"
    if os.path.isdir(d):
        os.makedirs(f"{RES}/{S}", exist_ok=True)
        for fn in ["submission.jsonl", "traces.jsonl"]:
            if os.path.exists(f"{d}/{fn}"): shutil.copy(f"{d}/{fn}", f"{RES}/{S}/{fn}")
open(f"{RES}/config.txt", "w").write(f"MODEL={MODEL}\nBACKEND={BACKEND}\nBATCH={BATCH}\nDTYPE={DTYPE}\nGPU={GPU_NAME}\n")
z = shutil.make_archive("/content/results", "zip", RES)
shutil.copy(z, os.path.join(DRIVE_DIR, "results.zip"))
print("Đã lưu", os.path.join(DRIVE_DIR, "results.zip"), round(os.path.getsize(z)/1e6, 1), "MB")
from google.colab import files
files.download(z)